# Kaggle - Brain Tumor MRI Dataset

You can find the dataset and some informations about on the [Kaggle page](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset).

For details on steps below, please see documentation in the *docs* directory.

## Kaggle notebook setup
### Installations

In [29]:
!pip install mlflow --quiet

### Data location

In [30]:
import os
print(os.listdir('/kaggle/input/'))
print(os.listdir('/kaggle/input/radimagenet-densenet121-notop'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed'))
print(os.listdir('/kaggle/input/brain-tumor-mri-preprocessed/processed'))

['radimagenet-densenet121-notop', 'brain-tumor-mri-preprocessed']
['RadImageNet-DenseNet121_notop.h5']
['processed']
['Validation', 'Training', 'Testing', '.gitkeep']


## General

In [73]:
import mlflow
import mlflow.tensorflow
from mlflow.tracking import MlflowClient
mlflow.set_tracking_uri("https://preachingly-nonabjuratory-marget.ngrok-free.dev")
mlflow.set_experiment("Brain_Tumor_Training")

<Experiment: artifact_location='mlflow-artifacts:/830660600119881173', creation_time=1769099937797, experiment_id='830660600119881173', last_update_time=1769099937797, lifecycle_stage='active', name='Brain_Tumor_Training', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [32]:
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime
import math

In [33]:
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))
tf.config.list_physical_devices('GPU')

Num GPUs Available: 1


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [71]:
# path management
PROJECT_ROOT = '/kaggle/input'
PREP_DIR = PROJECT_ROOT + '/brain-tumor-mri-preprocessed/processed'
ARTEFACTS_DIR = PROJECT_ROOT + '/radimagenet-densenet121-notop'

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
MODEL_DIR = "/kaggle/working/export_model"
os.makedirs(MODEL_DIR, exist_ok=True)

CLASSES = ["notumor", "glioma", "meningioma", "pituitary"]

# parameters
IMG_SIZE = 260
SEED = 42

PROJECT_NAME = "BrainTumorMRI"
MODEL_TYPE = "DenseNet121"
TWO_HEAD = True
MODEL_NAME = f"{PROJECT_NAME}_{MODEL_TYPE}_{TWO_HEAD*"2Head"}"
FREEZE_BACKBONE = True
MASK_TUMOR_TYPE_LOSS = True
BATCH_SIZE = 32
DATA_AUGMENTATION = True

## Modeling

### Backbone

In [35]:
# 1. Create DenseNet121 WITHOUT weights
backbone = DenseNet121(
    include_top=False,
    weights=None,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

# 2. Load RadImageNet weights
backbone.load_weights(ARTEFACTS_DIR + "/RadImageNet-DenseNet121_notop.h5")

# 3. Freeze the backbone for firsts training
backbone.trainable = not FREEZE_BACKBONE

print("✅ RadImageNet DenseNet121 loaded successfully")

✅ RadImageNet DenseNet121 loaded successfully


In [36]:
#backbone.summary()

In [37]:
w = backbone.weights[0].numpy()
print("Mean:", np.mean(w), "Std:", np.std(w))

Mean: -0.0028284893 Std: 0.10494729


Model seems to be correctly loaded.

### Model definition

In [38]:
model_data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomZoom((-0.03,0.03),(-0.03,0.03), seed=SEED),
    layers.RandomTranslation((-0.01,0.01),(-0.01,0.01), seed=SEED),
], name='data_augmentation_part')

In [39]:
model_head1 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(1,activation='sigmoid')
], name='tumor_presence')

In [40]:
model_head2 = keras.Sequential([
    layers.Dense(128, use_bias=False), # TO TEST : 256 ?
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.Dropout(0.2),
    layers.Dense(4,activation='softmax')
], name='tumor_type')

In [41]:
def shared_head_part(inputs, backbone, data_augmentation):
    # Data augmentation (training only)
    x = data_augmentation(inputs)
    # Backbone - force into inference
    x = backbone(x, training=False)

    # Shared head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)

    return x

In [42]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

x = shared_head_part(inputs, backbone, model_data_augmentation)

#Heads
output_presence = model_head1(x)
output_type = model_head2(x)

model = keras.Model(
    inputs=inputs,
    outputs={
        "tumor_presence": output_presence,
        "tumor_type": output_type
    },
    name='densenet_two_head'
)

In [43]:
loss_presence = keras.losses.BinaryFocalCrossentropy(
    gamma=2.0,
    alpha=0.25 # to favorize tumor detection (penalize false negatives), but taking account that tumors are 75% of data
)

In [78]:
#@keras.saving.register_keras_serializable()
@tf.keras.utils.register_keras_serializable()
def masked_sparse_cce(y_true, y_pred):
    tumor_present = tf.cast(y_true != 0, tf.float32)
    loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    loss = loss * tumor_present
    return tf.reduce_sum(loss) / (tf.reduce_sum(tumor_present) + 1e-6)

In [79]:
loss_weight_presence = 1.0
loss_weight_type = 1.3 # we give a little more weight to the classification of the type

model.compile(
    optimizer=keras.optimizers.Adam(), # change learning_rate for 1e-4 in fine-tuning steps
    loss={
        "tumor_presence": loss_presence,
        "tumor_type": masked_sparse_cce,
    },
    
    loss_weights={
        "tumor_presence": loss_weight_presence,
        "tumor_type": loss_weight_type, 
    },
    
    metrics={
        "tumor_presence": [
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.Precision(name="precision"),
            #keras.metrics.F1Score(name="f1_score"),
            keras.metrics.AUC(name="auc")
        ],
        "tumor_type": [
            "accuracy", 
            #"f1_score"
        ],
    }
)

In [46]:
#model.summary()

## Streaming Training

In [47]:
def parse_tfrecord(example_proto):
    """
    Parse a single TFRecord example and convert grayscale → RGB.
    """
    feature_description = {
        "image": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.int64),
    }

    example = tf.io.parse_single_example(example_proto, feature_description)

    # Deserialize image
    image = tf.io.parse_tensor(example["image"], out_type=tf.float32)

    # Shape after loading: (260, 260, 3)
    image.set_shape((260, 260, 3))

    label = tf.cast(example["label"], tf.int32)

    return image, label



def load_tfrecord_dataset(tfrecord_dir, shuffle=False, batch_size=1, repeat=False):
    """
    Load a TFRecord dataset from a directory.

    Args:
        tfrecord_dir (str or Path): Folder containing .tfrecord files
        shuffle (bool): Whether to shuffle files and samples
        batch_size (int): Batch size (can stay 1)
        repeat (bool): Repeat dataset indefinitely (for training)

    Returns:
        tf.data.Dataset
    """
    tfrecord_files = tf.io.gfile.glob(
        str(tfrecord_dir) + "/*.tfrecord"
    )

    ds = tf.data.TFRecordDataset(
        tfrecord_files,
        num_parallel_reads=tf.data.AUTOTUNE
    )

    ds = ds.map(
        parse_tfrecord,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    if shuffle:
        ds = ds.shuffle(buffer_size=512)

    if repeat:
        ds = ds.repeat()

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


In [48]:
TRAIN_DIR = PREP_DIR + "/Training"
VAL_DIR   = PREP_DIR + "/Validation"
TEST_DIR  = PREP_DIR + "/Testing"

train_ds = load_tfrecord_dataset(
    TRAIN_DIR,
    shuffle=True,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

val_ds = load_tfrecord_dataset(
    VAL_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)

"""
test_ds = load_tfrecord_dataset(
    TEST_DIR,
    shuffle=False,
    batch_size=BATCH_SIZE,
    repeat=False
).prefetch(tf.data.AUTOTUNE)
"""

'\ntest_ds = load_tfrecord_dataset(\n    TEST_DIR,\n    shuffle=False,\n    batch_size=BATCH_SIZE,\n    repeat=False\n).prefetch(tf.data.AUTOTUNE)\n'

In [49]:
def split_labels(image, label):
    """
    Create labels for a 2-head model.
    """
    tumor_present = tf.cast(label != 0, tf.float32)
    tumor_type = tf.cast(label, tf.int32)

    return image, {
        "tumor_presence": tumor_present,
        "tumor_type": tumor_type
    }


In [50]:
train_ds = train_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(split_labels, num_parallel_calls=tf.data.AUTOTUNE)

In [51]:
#for x, y in train_ds.take(1):
#    print("Image:")
#    print(x.dtype, x.shape)
#    print("\nLabels:")
#    for k, v in y.items():
#        print(k, v.dtype, v.shape)

In [52]:
def count_tfrecord_batches(directory_path, batch_size):
    """
    Count number of batches for a TFRecord dataset.

    Assumes:
    - 1 sample per TFRecord
    - batch_size is variable

    Returns:
    - number of batches = ceil(num_samples / batch_size)
    """
    directory_path = Path(directory_path)
    n_samples = len(list(directory_path.glob("*.tfrecord")))
    return math.ceil(n_samples / batch_size)


In [53]:
print(train_ds.element_spec)

(TensorSpec(shape=(None, 260, 260, 3), dtype=tf.float32, name=None), {'tumor_presence': TensorSpec(shape=(None,), dtype=tf.float32, name=None), 'tumor_type': TensorSpec(shape=(None,), dtype=tf.int32, name=None)})


In [54]:
#to_monitor = "val_tumor_presence_recall"
#mode = "max"
to_monitor = "val_tumor_type_loss"
mode = "min"

reduce_lr = ReduceLROnPlateau(
    monitor=to_monitor,
    mode=mode,
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor=to_monitor,
    mode=mode,
    min_delta=0.0001,
    patience=10,
    restore_best_weights=False,
    verbose=1,
)

checkpoint_cb = keras.callbacks.ModelCheckpoint(
    filepath=CHECKPOINT_DIR + "/epoch_{epoch:02d}.weights.h5",
    monitor=to_monitor,
    mode=mode,
    save_best_only=False,
    save_weights_only=True,
    verbose=1,
)

terminate_nan = keras.callbacks.TerminateOnNaN()

In [55]:
#print("RUN_NAME:", RUN_NAME)
#print("TRAIN steps:", count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE))
#print("VAL steps:", count_tfrecord_batches(VAL_DIR, BATCH_SIZE))
#print("train_ds element_spec:", train_ds.element_spec)

In [56]:
raise Exception("Do not fit from scratch again. Use the best head model !")

RUN_NAME = (
    f"{MODEL_TYPE}"
    f"freeze={FREEZE_BACKBONE}_"
    f"mask={MASK_TUMOR_TYPE_LOSS}_"
    f"{datetime.now().strftime('%Y%m%d-%H%M')}"
)

print(f"Run name: {RUN_NAME}\n")

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.tensorflow.autolog(registered_model_name=MODEL_NAME)

    mlflow.log_params({
        "project": PROJECT_NAME,
        "model_type": MODEL_TYPE,
        "pretrained_weights": "RadImageNet",
        "backbone_frozen": FREEZE_BACKBONE,
        "head_1": "tumor_presence_binary",
        "head_2": "tumor_type_softmax",
        "loss_weight_presence": loss_weight_presence,
        "loss_weight_type": loss_weight_type,
        "mask_tumor_type_loss": MASK_TUMOR_TYPE_LOSS,
        "label_0_means": "no_tumor",
        "data_augmentation": DATA_AUGMENTATION,
    })

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=60,
        #steps_per_epoch=count_tfrecord_batches(TRAIN_DIR, BATCH_SIZE),
        #validation_steps=count_tfrecord_batches(VAL_DIR, BATCH_SIZE),
        callbacks=[reduce_lr, early_stopping, checkpoint_cb, terminate_nan],
        verbose=1,
    )

    client = MlflowClient()
    latest_version = client.get_latest_versions(MODEL_NAME)[-1].version

    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=latest_version,
        stage="Staging"
    )


Run name: DenseNet121freeze=True_mask=True_20260204-1641



2026/02/04 16:41:06 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.
2026/02/04 16:41:08 WARNING mlflow.data.tensorflow_dataset: Failed to infer schema for TensorFlow dataset. Exception: Failed to infer schema for tf.data.Dataset. Schemas can only be inferred if the dataset consists of tensors. Ragged tensors, tensor arrays, and other types are not supported. Additionally, datasets with nested tensors are not supported.


Epoch 1/60
    143/Unknown 36s 143ms/step - loss: 1.2487 - tumor_presence_accuracy: 0.7790 - tumor_presence_auc: 0.8119 - tumor_presence_loss: 0.1504 - tumor_presence_precision: 0.8613 - tumor_presence_recall: 0.8285 - tumor_type_accuracy: 0.4994 - tumor_type_loss: 0.8449
Epoch 1: saving model to /kaggle/working/checkpoints/epoch_01.weights.h5


143/143 ━━━━━━━━━━━━━━━━━━━━ 68s 363ms/step - loss: 1.2470 - tumor_presence_accuracy: 0.7794 - tumor_presence_auc: 0.8124 - tumor_presence_loss: 0.1501 - tumor_presence_precision: 0.8615 - tumor_presence_recall: 0.8289 - tumor_type_accuracy: 0.4997 - tumor_type_loss: 0.8437 - val_loss: 1.5131 - val_tumor_presence_accuracy: 0.3185 - val_tumor_presence_auc: 0.9626 - val_tumor_presence_loss: 0.4738 - val_tumor_presence_precision: 1.0000 - val_tumor_presence_recall: 0.0546 - val_tumor_type_accuracy: 0.4339 - val_tumor_type_loss: 0.7674 - learning_rate: 0.0010
Epoch 2/60
143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - loss: 0.7863 - tumor_presence_accuracy: 0.8853 - tumor_presence_auc: 0.9320 - tumor_presence_loss: 0.0805 - tumor_presence_precision: 0.9147 - tumor_presence_recall: 0.9275 - tumor_type_accuracy: 0.5530 - tumor_type_loss: 0.5429
Epoch 2: saving model to /kaggle/working/checkpoints/epoch_02.weights.h5
143/143 ━━━━━━━━━━━━━━━━━━━━ 26s 178ms/step - loss: 0.7860 - tumor_presence_accu

143/143 ━━━━━━━━━━━━━━━━━━━━ 48s 330ms/step - loss: 0.6282 - tumor_presence_accuracy: 0.9084 - tumor_presence_auc: 0.9549 - tumor_presence_loss: 0.0662 - tumor_presence_precision: 0.9313 - tumor_presence_recall: 0.9433 - tumor_type_accuracy: 0.5960 - tumor_type_loss: 0.4323 - val_loss: 1.2365 - val_tumor_presence_accuracy: 0.9055 - val_tumor_presence_auc: 0.9529 - val_tumor_presence_loss: 0.0611 - val_tumor_presence_precision: 0.9032 - val_tumor_presence_recall: 0.9733 - val_tumor_type_accuracy: 0.3500 - val_tumor_type_loss: 0.8802 - learning_rate: 0.0010
Epoch 4/60
143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 144ms/step - loss: 0.6281 - tumor_presence_accuracy: 0.9118 - tumor_presence_auc: 0.9644 - tumor_presence_loss: 0.0580 - tumor_presence_precision: 0.9407 - tumor_presence_recall: 0.9383 - tumor_type_accuracy: 0.5895 - tumor_type_loss: 0.4385
Epoch 4: saving model to /kaggle/working/checkpoints/epoch_04.weights.h5
143/143 ━━━━━━━━━━━━━━━━━━━━ 26s 180ms/step - loss: 0.6281 - tumor_presence_accu

...

143/143 ━━━━━━━━━━━━━━━━━━━━ 69s 484ms/step - loss: 0.2765 - tumor_presence_accuracy: 0.9733 - tumor_presence_auc: 0.9964 - tumor_presence_loss: 0.0185 - tumor_presence_precision: 0.9841 - tumor_presence_recall: 0.9790 - tumor_type_accuracy: 0.6706 - tumor_type_loss: 0.1985 - val_loss: 0.3252 - val_tumor_presence_accuracy: 0.9790 - val_tumor_presence_auc: 0.9956 - val_tumor_presence_loss: 0.0185 - val_tumor_presence_precision: 0.9902 - val_tumor_presence_recall: 0.9806 - val_tumor_type_accuracy: 0.6465 - val_tumor_type_loss: 0.2304 - learning_rate: 3.1250e-05
Epoch 42/60
143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 142ms/step - loss: 0.2858 - tumor_presence_accuracy: 0.9785 - tumor_presence_auc: 0.9969 - tumor_presence_loss: 0.0168 - tumor_presence_precision: 0.9870 - tumor_presence_recall: 0.9834 - tumor_type_accuracy: 0.6599 - tumor_type_loss: 0.2070
Epoch 42: saving model to /kaggle/working/checkpoints/epoch_42.weights.h5


143/143 ━━━━━━━━━━━━━━━━━━━━ 58s 405ms/step - loss: 0.2857 - tumor_presence_accuracy: 0.9785 - tumor_presence_auc: 0.9969 - tumor_presence_loss: 0.0168 - tumor_presence_precision: 0.9870 - tumor_presence_recall: 0.9834 - tumor_type_accuracy: 0.6599 - tumor_type_loss: 0.2069 - val_loss: 0.3004 - val_tumor_presence_accuracy: 0.9808 - val_tumor_presence_auc: 0.9955 - val_tumor_presence_loss: 0.0185 - val_tumor_presence_precision: 0.9890 - val_tumor_presence_recall: 0.9842 - val_tumor_type_accuracy: 0.6579 - val_tumor_type_loss: 0.2116 - learning_rate: 3.1250e-05
Epoch 43/60
143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 0.2645 - tumor_presence_accuracy: 0.9739 - tumor_presence_auc: 0.9963 - tumor_presence_loss: 0.0185 - tumor_presence_precision: 0.9869 - tumor_presence_recall: 0.9770 - tumor_type_accuracy: 0.6655 - tumor_type_loss: 0.1892
Epoch 43: saving model to /kaggle/working/checkpoints/epoch_43.weights.h5
143/143 ━━━━━━━━━━━━━━━━━━━━ 26s 180ms/step - loss: 0.2645 - tumor_presenc

143/143 ━━━━━━━━━━━━━━━━━━━━ 78s 542ms/step - loss: 0.2387 - tumor_presence_accuracy: 0.9814 - tumor_presence_auc: 0.9969 - tumor_presence_loss: 0.0168 - tumor_presence_precision: 0.9883 - tumor_presence_recall: 0.9859 - tumor_type_accuracy: 0.6667 - tumor_type_loss: 0.1707 - val_loss: 0.2990 - val_tumor_presence_accuracy: 0.9825 - val_tumor_presence_auc: 0.9958 - val_tumor_presence_loss: 0.0179 - val_tumor_presence_precision: 0.9902 - val_tumor_presence_recall: 0.9854 - val_tumor_type_accuracy: 0.6544 - val_tumor_type_loss: 0.2174 - learning_rate: 1.5625e-05
Epoch 51/60
143/143 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 0.2618 - tumor_presence_accuracy: 0.9778 - tumor_presence_auc: 0.9970 - tumor_presence_loss: 0.0164 - tumor_presence_precision: 0.9849 - tumor_presence_recall: 0.9844 - tumor_type_accuracy: 0.6672 - tumor_type_loss: 0.1887
Epoch 51: saving model to /kaggle/working/checkpoints/epoch_51.weights.h5
143/143 ━━━━━━━━━━━━━━━━━━━━ 26s 179ms/step - loss: 0.2617 - tumor_presenc

2026/02/04 17:12:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/04 17:12:35 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmpxkgyxukx/model, flavor: tensorflow). Fall back to return ['tensorflow==2.19.0', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 
Registered model 'BrainTumorMRI_DenseNet121_2Head' already exists. Creating a new version of this model...
2026/02/04 17:12:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BrainTumorMRI_DenseNet121_2Head, version 19
Created version '19' of model 'BrainTumorMRI_DenseNet121_2Head'.
/tmp/ipykernel_55/1980140685.py:39: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecat

🏃 View run DenseNet121freeze=True_mask=True_20260204-1641 at: https://preachingly-nonabjuratory-marget.ngrok-free.dev/#/experiments/830660600119881173/runs/5557c30f1d2540b4ae30afa7f1d8b4f8
🧪 View experiment at: https://preachingly-nonabjuratory-marget.ngrok-free.dev/#/experiments/830660600119881173


## Epoch filter

In [57]:
history_df = pd.DataFrame(history.history)
history_df["epoch"] = history_df.index
#history_df.head(5)

In [58]:
metrics_cols = [
    "epoch",
    "val_tumor_presence_recall",
    "val_tumor_type_accuracy",
    "val_tumor_presence_loss",
    "val_tumor_type_loss"
]

df = history_df[metrics_cols].copy()

In [59]:
df = df[
    (df["val_tumor_presence_recall"] >= 0.94) &
    (df["val_tumor_type_accuracy"] >= 0.55)
]
#df

,epoch,val_tumor_presence_recall,val_tumor_type_accuracy,val_tumor_presence_loss,val_tumor_type_loss
12,12,0.995146,0.643920,0.037375,0.291258
13,13,0.984223,0.652668,0.023051,0.280522
15,15,0.997573,0.563430,0.046265,0.489436
17,17,0.987864,0.643920,0.023655,0.276753
24,24,0.981796,0.595801,0.024683,0.370762
25,25,0.945388,0.595801,0.030368,0.489721
26,26,0.985437,0.622922,0.019657,0.298615
27,27,0.987864,0.654418,0.020886,0.240111
28,28,0.951456,0.606299,0.028617,0.344925
29,29,0.952670,0.657043,0.027053,0.244568


In [60]:
def normalize(col):
    return (col - col.min()) / (col.max() - col.min() + 1e-8)

df["pres_rec_norm"] = normalize(df["val_tumor_presence_recall"])
df["type_accu_norm"] = normalize(df["val_tumor_type_accuracy"])
df["pres_loss_norm"] = 1 - normalize(df["val_tumor_presence_loss"])
df["type_loss_norm"] = 1 - normalize(df["val_tumor_type_loss"])

In [61]:
df["S"] = (
    0.40 * df["pres_rec_norm"]
  + 0.35 * df["type_accu_norm"]
  + 0.15 * df["pres_loss_norm"]
  + 0.10 * df["type_loss_norm"]
)
#df

,epoch,val_tumor_presence_recall,val_tumor_type_accuracy,val_tumor_presence_loss,val_tumor_type_loss,pres_rec_norm,type_accu_norm,pres_loss_norm,type_loss_norm,S
12,12,0.995146,0.643920,0.037375,0.291258,0.953487,0.836364,3.068861e-01,7.135993e-01,0.791515
13,13,0.984223,0.652668,0.023051,0.280522,0.744186,0.927273,8.013507e-01,7.522013e-01,0.817642
15,15,0.997573,0.563430,0.046265,0.489436,1.000000,0.000000,3.452097e-07,1.022647e-03,0.400102
17,17,0.987864,0.643920,0.023655,0.276753,0.813953,0.836364,7.804953e-01,7.657535e-01,0.811958
24,24,0.981796,0.595801,0.024683,0.370762,0.697674,0.336363,7.450067e-01,4.277307e-01,0.551321
25,25,0.945388,0.595801,0.030368,0.489721,0.000000,0.336363,5.487592e-01,3.595632e-08,0.200041
26,26,0.985437,0.622922,0.019657,0.298615,0.767442,0.618181,9.185372e-01,6.871437e-01,0.729835
27,27,0.987864,0.654418,0.020886,0.240111,0.813953,0.945454,8.761143e-01,8.975031e-01,0.877658
28,28,0.951456,0.606299,0.028617,0.344925,0.116278,0.445454,6.092224e-01,5.206315e-01,0.345867
29,29,0.952670,0.657043,0.027053,0.244568,0.139535,0.972727,6.631980e-01,8.814796e-01,0.583896


In [62]:
best_row = df.sort_values("S", ascending=False).iloc[0]
best_epoch = int(best_row["epoch"])

print(f"✅ Best epoch selected from S: {best_epoch}")
print(best_row)

✅ Best epoch selected from S: 43
epoch                        43.000000
val_tumor_presence_recall     0.985437
val_tumor_type_accuracy       0.659668
val_tumor_presence_loss       0.018888
val_tumor_type_loss           0.211857
pres_rec_norm                 0.767442
type_accu_norm                1.000000
pres_loss_norm                0.945080
type_loss_norm                0.999096
S                             0.898648
Name: 43, dtype: float64


In [68]:
mlflow.set_tags({
    "model_stage": "best_manual_epoch",
    "best_epoch": best_epoch,
    "selection_method": "composite_score_S",
})

mlflow.log_metric("S", best_row.iloc[-1])
mlflow.log_metric("Best epoch", best_row.iloc[0])

In [69]:
model.load_weights(f"{CHECKPOINT_DIR}/epoch_{best_epoch:02d}.weights.h5")
print(f"✅ Loaded best epoch: {best_epoch}")

✅ Loaded best epoch: 43


In [70]:
mlflow.tensorflow.log_model(
    model,
    name=f"best_epoch_{best_epoch}_manual",
    registered_model_name=MODEL_NAME
)
print(f"✅ Registered best model: {MODEL_NAME}")

2026/02/04 17:46:33 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/02/04 17:46:45 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /tmp/tmp1fggz53y/model, flavor: tensorflow). Fall back to return ['tensorflow==2.19.0', 'cloudpickle==3.1.1']. Set logging level to DEBUG to see the full traceback. 
Registered model 'BrainTumorMRI_DenseNet121_2Head' already exists. Creating a new version of this model...
2026/02/04 17:46:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: BrainTumorMRI_DenseNet121_2Head, version 22
Created version '22' of model 'BrainTumorMRI_DenseNet121_2Head'.


✅ Registered best model: BrainTumorMRI_DenseNet121_2Head


In [72]:
model.save(f"{MODEL_DIR}/brain_tumor_model_best_epoch_{best_epoch}.keras")
model.save_weights(
    f"{MODEL_DIR}/brain_tumor_weights_epoch_{best_epoch}.weights.h5"
)
mlflow.log_artifacts(MODEL_DIR, artifact_path="exported_model_files")

### Loading final model from MLFlow

In [80]:
model = mlflow.tensorflow.load_model(
    "models:/BrainTumorMRI_DenseNet121_2Head/latest"
)
print("✅ Model loaded successfully with custom loss")

✅ Model loaded successfully with custom loss


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'loss_scale_optimizer', because it has 19 variables whereas the saved optimizer has 32 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 15 variables whereas the saved optimizer has 28 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [81]:
model.summary()

Model: "densenet_two_head"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 260, 260,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ data_augmentation_… │ (None, 260, 260,  │          0 │ input_layer_6[0]… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ densenet121         │ (None, 8, 8,      │  7,037,504 │ data_augmentatio… │
│ (Functional)        │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 1024)      │          0 │ densenet121[0][0] │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 512)       │    524,288 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 512)       │      2,048 │ dense_9[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 512)       │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 512)       │          0 │ activation_5[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tumor_presence      │ (None, 1)         │     66,177 │ dropout_5[0][0]   │
│ (Sequential)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tumor_type          │ (None, 4)         │     66,564 │ dropout_5[0][0]   │
│ (Sequential)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 8,354,128 (31.87 MB)

 Trainable params: 657,541 (2.51 MB)

 Non-trainable params: 7,039,040 (26.85 MB)

 Optimizer params: 657,547 (2.51 MB)

In [82]:
model.evaluate(val_ds)

36/36 ━━━━━━━━━━━━━━━━━━━━ 12s 120ms/step - loss: 0.3289 - tumor_presence_accuracy: 0.9850 - tumor_presence_auc: 0.9958 - tumor_presence_loss: 0.0168 - tumor_presence_precision: 0.9880 - tumor_presence_recall: 0.9916 - tumor_type_accuracy: 0.6632 - tumor_type_loss: 0.2400


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


[0.3676888942718506,
 0.017765367403626442,
 0.2637573480606079,
 0.9816272854804993,
 0.9955679774284363,
 0.9890377521514893,
 0.9854369163513184,
 0.6342957019805908]

### Confusion Matrix

### Grad-CAM

## Fine-Tuning

In [66]:
Warning : do not forget :
- mtrx confu classif
- XAI (Grad-CAM ?)
- stratégie de fine-tuning du backbone
- BatchNormalization precaution in fine-tuning

SyntaxError: invalid syntax (560267976.py, line 1)